In [1]:
%load_ext aiida
%aiida

Loaded AiiDA DB environment - profile name: default.

In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import re
import numpy as np
import matplotlib.pyplot as plt

In [3]:
import re
import numpy as np


def extract_band_energies(output_text):

    float_re = re.compile(r"[-+]?\d*\.\d+|[-+]?\d+")

    lines = output_text.splitlines()

    withspin=False
    number_electrons = None
    n_up = None
    n_down = None

    Ef_nscf = None
    Ef_scf = None

    spin_up_vals = []
    spin_down_vals = []
    nonspin_vals = []

    current_spin = None
    inside_band = False

    for i, line in enumerate(lines):
        low = line.lower()

        # ---------------- ELECTRONS ----------------
        if "number of electrons" in low:
            vals = float_re.findall(line)
            number_electrons = float(vals[0])

            if "up:" in low:
                n_up = int(float(vals[1]))
                n_down = int(float(vals[2]))
                withspin = True

        # ---------------- FERMI ----------------
        if "spin up/dw fermi energies are" in low:
            vals = float_re.findall(line)
            Ef_nscf = np.array([float(vals[0]), float(vals[1])])
            withspin = True

            if i + 1 < len(lines) and "compare with" in lines[i + 1].lower():
                vals2 = float_re.findall(lines[i + 1])
                Ef_scf = np.array([float(vals2[0]), float(vals2[1])])

        elif "the fermi energy is" in low:
            vals = float_re.findall(line)
            Ef_nscf = float(vals[0])

            if i + 1 < len(lines) and "compare with" in lines[i + 1].lower():
                vals2 = float_re.findall(lines[i + 1])
                Ef_scf = float(vals2[0])

        # ---------------- SPIN BLOCKS ----------------
        if "------ spin up" in low:
            current_spin = "up"
            continue

        if "------ spin down" in low:
            current_spin = "down"
            continue

        # ---------------- BAND BLOCK ----------------
        if "bands (ev)" in low:
            inside_band = True
            continue

        if inside_band and "occupation numbers" in low:
            inside_band = False
            continue

        if inside_band:
            nums = float_re.findall(line)
            if nums:
                vals = [float(x) for x in nums]
                if current_spin == "up":
                    spin_up_vals.extend(vals)
                elif current_spin == "down":
                    spin_down_vals.extend(vals)
                else:
                    nonspin_vals.extend(vals)

    if number_electrons is None:
        raise ValueError("Electrons not found.")

    if Ef_nscf is None:
        raise ValueError("Fermi energy not found.")

    # ---------------- CONSTRUCT ARRAYS ----------------
    if withspin:
        
        eig_up = np.array(spin_up_vals)
        eig_down = np.array(spin_down_vals)

        eigenvalues = np.array([
            eig_up[np.newaxis, :],
            eig_down[np.newaxis, :]
        ])

        N = 1
        
        HOMO = np.zeros(2)
        LUMO = np.zeros(2)

        HOMO[0] = eig_up[n_up - 1]
        LUMO[0] = eig_up[n_up]

        HOMO[1] = eig_down[n_down - 1]
        LUMO[1] = eig_down[n_down]

    else:

        eig = np.array(nonspin_vals)[np.newaxis, :]
        eigenvalues = eig
        N = 1

        nelec_half = int(number_electrons / 2)

        HOMO = eig[0, nelec_half - 1]
        LUMO = eig[0, nelec_half]

    return withspin,Ef_nscf, Ef_scf, number_electrons, N, eigenvalues, HOMO, LUMO

In [4]:
import matplotlib.pyplot as plt


def plot_dos(
    eigenvalues,
    Ef_nscf,
    HOMO,
    LUMO,
    sigma=0.05,
    energy_range=None,
    bin_size=0.05,
    withspin=False
):

                
    if withspin:
        Ef_shift = float(np.max(np.asarray(Ef_nscf)))
        eig_up = eigenvalues[0].flatten() - Ef_shift
        eig_dn = eigenvalues[1].flatten() - Ef_shift
    else:
        Ef_shift = float(Ef_nscf)
        eig = eigenvalues.flatten() - Ef_shift        
        
        

    if energy_range is None:
        if withspin:
            emin = min(eig_up.min(), eig_dn.min())
            emax = max(eig_up.max(), eig_dn.max())
        else:
            emin = eig.min()
            emax = eig.max()
    else:
        emin, emax = energy_range

    grid = np.arange(emin, emax + bin_size, bin_size)

    def gaussian_dos(eigs):
        dos = np.zeros_like(grid)
        pref = 1.0 / (sigma * np.sqrt(2 * np.pi))
        for e in eigs:
            dos += pref * np.exp(-0.5 * ((grid - e) / sigma) ** 2)
        return dos

    plt.figure(figsize=(7, 5))

    if withspin:
        plt.plot(grid, gaussian_dos(eig_up), label="dos up")
        plt.plot(grid, -gaussian_dos(eig_dn), label="dos down")

        plt.axvline(HOMO[0] - Ef_shift, linestyle="--", color="red")
        plt.axvline(LUMO[0] - Ef_shift, linestyle="--", color="blue")
        plt.axvline(HOMO[1] - Ef_shift, linestyle="--", color="red")
        plt.axvline(LUMO[1] - Ef_shift, linestyle="--", color="blue")
    else:
        plt.plot(grid, gaussian_dos(eig), label="dos")
        plt.axvline(HOMO - Ef_shift, linestyle="--", color="red")
        plt.axvline(LUMO - Ef_shift, linestyle="--", color="blue")

    plt.axvline(0.0, color="black", linewidth=1)
    plt.xlabel("Energy (eV)  (Ef = 0)")
    plt.ylabel("DOS (arb. units)")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [5]:
# -----------------------------
# Helper: parse energy range
# -----------------------------
def parse_energy_range(text):
    text = text.strip()
    if text.lower() == "full" or text == "":
        return None

    numbers = re.findall(r"[-+]?\d*\.\d+|[-+]?\d+", text)
    if len(numbers) != 2:
        raise ValueError("Energy range must contain exactly two numbers or Full.")

    return [float(numbers[0]), float(numbers[1])]


# -----------------------------
# Widgets
# -----------------------------
pk_widget = widgets.IntText(
    description="QE calc PK",
    value=0,
)

sigma_widget = widgets.FloatText(
    description="Gaussian σ (eV)",
    value=0.02,
)

bin_widget = widgets.FloatText(
    description="Bin size (eV)",
    value=0.01,
)

range_widget = widgets.Text(
    description="Energy range",
    value="Full",
)

plot_button = widgets.Button(
    description="Make plot",
    button_style="primary",
)

output_area = widgets.Output()


# -----------------------------
# Button callback
# -----------------------------
def on_button_clicked(b):
    with output_area:
        clear_output()

        try:
            pk = pk_widget.value
            sigma = sigma_widget.value
            bin_size = bin_widget.value
            energy_range = parse_energy_range(range_widget.value)

            qe = load_node(pk)

            output = None
            withspin = False

            for n in qe.called_descendants:

                thenscf = (
                    n.process_label == "PwCalculation"
                    and n.inputs.parameters.get_dict()["CONTROL"]["calculation"] == "nscf"
                )

                thenscf = thenscf and (
                    n.caller.caller.process_label in ["PdosWorkChain"]
                )

                if thenscf:
                    output = n.outputs.retrieved.get_object_content("aiida.out")

            if output is None:
                raise RuntimeError("No NSCF PwCalculation found.")

            withspin, Ef_nscf, Ef_scf, ne, nptk, eigenvalues, HOMO, LUMO = \
                extract_band_energies(output)

            plot_dos(
                eigenvalues,
                Ef_nscf,
                HOMO,
                LUMO,
                sigma=sigma,
                energy_range=energy_range,
                bin_size=bin_size,
                withspin=withspin
            )

            print("\n=== Fermi energies ===")
            print("NSCF:", Ef_nscf)
            print("SCF :", Ef_scf)


            print("\n=== HOMO / LUMO relative to max(NSCF Ef) ===")

            Ef_shift = Ef_nscf.max() if withspin else Ef_nscf

            if withspin:
                print(
                    f"Spin up   HOMO: {(HOMO[0] - Ef_shift):.2f}   "
                    f"LUMO: {(LUMO[0] - Ef_shift):.2f}"
                )
                print(
                    f"Spin down HOMO: {(HOMO[1] - Ef_shift):.2f}   "
                    f"LUMO: {(LUMO[1] - Ef_shift):.2f}"
                )
            else:
                print(
                    f"HOMO: {(HOMO - Ef_shift):.2f}   "
                    f"LUMO: {(LUMO - Ef_shift):.2f}"
                )

        except Exception as e:
            print("Error:", e)


plot_button.on_click(on_button_clicked)


# -----------------------------
# Display UI
# -----------------------------
display(
    widgets.VBox(
        [
            pk_widget,
            sigma_widget,
            bin_widget,
            range_widget,
            plot_button,
            output_area,
        ]
    )
)